# 🎓 1-Hour RAG Crash Course: Real-World Architecture & Live Code
### *A Practical, Engineer-to-Engineer Training on Building Production RAG*

> **Trainer's Playbook**: Welcome! This notebook is designed for a **1-hour practical team workshop**. 
> Instead of getting lost in heavy academic math, we focus on **software engineering reality**: real architectural tradeoffs, real failure modes, intuitive analogies, and live executable code you can run on screen.

---

### ⏱️ The 60-Minute Workshop Schedule

| Time Budget | Topic | The Intuitive Analogy | What We Code Live |
|---|---|---|---|
| **00:00 – 00:08 (8m)** | **1. Why RAG? (The Problem)** | Open-Book vs Closed-Book Exam | Plain LLM Hallucinating vs RAG Grounded Answer |
| **00:08 – 00:18 (10m)**| **2. Chunking Done Right** | The Index Card vs The Whole Book | The "Parent-Child" trick (Search with index, read paragraph) |
| **00:18 – 00:28 (10m)**| **3. Vector Embeddings** | GPS Coordinates for Human Meaning | Proving two sentences with 0 words in common cluster together |
| **00:28 – 00:38 (10m)**| **4. Hybrid Search (Dense + BM25)** | The Fuzzy Artist + The Exact Accountant | Why Vector Search fails on Drug Names / SKUs & how RRF fixes it |
| **00:38 – 00:46 (8m)** | **5. Reranking (Bi vs Cross-Encoder)**| The Quick Resume Scan vs The In-Person Interview | Catching "NOT" / Contraindications with local MiniLM CPU model |
| **00:46 – 00:54 (8m)** | **6. Guardrails & The "Bouncer"** | The Club Bouncer (The Confidence Gate) | Blocking chocolate cake / off-topic queries in 0.1ms for $0.00 |
| **00:54 – 01:00 (6m)** | **7. How to Measure Success** | The 3 Metrics Every Tech Lead Needs (RAG Triad) | Live Q&A and Production Architecture Cheat Sheet |


---
## ⚙️ Step 0: Quick Environment Check
Let's check our setup and load our OpenAI API key.


In [ ]:
import os
import sys
import json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

# Load API keys from root .env or local .env
load_dotenv(Path("../.env"))
load_dotenv(Path(".env"))

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️ WARNING: OPENAI_API_KEY not found in .env! Please add it to test live OpenAI calls.")
else:
    print("✅ Ready! OpenAI API key loaded:", api_key[:8] + "..." + api_key[-4:])


---
## 🧠 Module 1: Why RAG? (00:00 – 00:08)
### *Closed-Book vs Open-Book: Why Smart LLMs Make Dumb Mistakes*

### The Mental Model: The University Student
- **A Plain LLM is a student taking a CLOSED-BOOK exam**:
  - They read billions of books 6 months ago during training.
  - If you ask about general knowledge ("Explain photosynthesis"), they do great.
  - If you ask about **your company's private policy**, **recent updates**, or **specific medicine doses**, they panic and make up a plausible-sounding lie (**Hallucination**).
- **RAG is an OPEN-BOOK exam**:
  - You don't ask the student to memorize your 1,000-page internal PDF manual.
  - You let a smart search engine find the exact 2 relevant paragraphs, hand them to the student, and say: *"Answer using ONLY these 2 paragraphs, and cite your page number."*

### Why not just stuff the whole PDF into the prompt?
1. **Cost**: 100-page PDF $\approx$ 50,000 tokens $\approx$ $0.15 per question. At 10,000 queries/day = **$45,000/month**!
2. **Speed**: Processing 50k tokens takes 8–15 seconds per response.
3. **"Lost in the Middle"**: LLMs get confused and miss facts buried in the middle of giant prompts.


In [ ]:
# LIVE DEMO: Plain LLM (Blind Guess) vs RAG (Open-Book Grounding)
from openai import OpenAI
client = OpenAI()

question = "What is the specific 2nd-day medication given for gametocyte clearance in falciparum malaria in India?"

print("--- 1. WHAT A PLAIN LLM DOES (Closed-Book) ---")
naive_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": question}],
    temperature=0.7,
    max_tokens=120
)
print(naive_response.choices[0].message.content)
print("\n" + "="*70 + "\n")

print("--- 2. WHAT RAG DOES (Open-Book with Retrieved Context) ---")
# Pretend our search engine retrieved this exact snippet from official guidelines:
retrieved_snippet = (
    "[GUIDELINE: malaria.pdf, Page 6]\n"
    "For uncomplicated P. falciparum malaria across India, Artemisinin Combination Therapy (ACT) is given. "
    "On Day 2, a single dose of Primaquine (0.75 mg/kg body weight) must be administered for gametocyte "
    "clearance to prevent transmission. Primaquine is strictly contraindicated in pregnancy and infants under 1 year."
)

rag_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a factual assistant. Answer ONLY using the provided text. Cite the source."},
        {"role": "user", "content": f"CONTEXT:\n{retrieved_snippet}\n\nQUESTION: {question}"}
    ],
    temperature=0.1
)
print(rag_response.choices[0].message.content)


---
## 📑 Module 2: The Chunking Trap & The Parent-Child Secret (00:08 – 00:18)
### *Why "Flat Chunking" Breaks in Production*

### The Real-World Engineering Dilemma:
When you ingest a PDF, you must split it into chunks. How big should a chunk be?

| Chunk Size | What Happens in Search | What Happens in Generation | Real-World Danger |
|---|---|---|---|
| **Too Big** (1,500 words) | ❌ Embedding is a vague "soup". Precision drops. | ✅ Plenty of context for the LLM. | Can't find specific numbers, clauses, or SKUs. |
| **Too Small** (50 words) | ✅ Search matches the exact 5 words easily! | ❌ LLM misses prerequisites & warnings. | **LETHAL**: Finds *"Take 500mg"* but misses *"Do not take if pregnant"*! |

---

### The Practical Solution: Hierarchical Parent-Child Chunking
Think of a **Database Index**:
- The **Index** is tiny and fast (just an ID or a keyword).
- The **Table Row** contains the full customer record!

We do the same thing with text:
1. We create a **Parent Chunk** (1,200 characters $\approx$ 250 words) — contains the full story, warnings, prerequisites.
2. We slice that parent into small **Child Chunks** (250 characters $\approx$ 50 words).
3. **The Trick**: We embed the **Child** into the search engine. But in its metadata, we store `parent_text`.
4. When a user searches, the Child hits the query with pinpoint accuracy... but we **hand the LLM the Parent**!

```
[Raw Document Paragraph (1,200 chars: Parent)]
 ├── Child 1 (250 chars) ──► Embed into Vector DB (Search Key)
 ├── Child 2 (250 chars) ──► Embed into Vector DB (Search Key)
 └── Child 3 (250 chars) ──► Embed into Vector DB (Search Key)

At Query Time:
User searches: "artesunate dose" ──► Matches Child 1!
Retriever swaps Child 1 ─────────► Hands LLM Parent (with pregnancy warnings intact!)
```


In [ ]:
# LIVE DEMO: Building Parent-Child Chunks in 10 Lines of Python
from langchain_text_splitters import RecursiveCharacterTextSplitter
import hashlib

clinical_manual_text = (
    "In pregnant patients diagnosed with severe falciparum malaria during the first trimester, "
    "parenteral artesunate is the drug of choice over quinine. Intravenous artesunate should be "
    "administered at 2.4 mg/kg body weight on admission, repeated at 12 hours and 24 hours. "
    "Thereafter, once the patient tolerates oral fluids, oral artemisinin-based combination therapy (ACT) "
    "should follow for 3 full days. Full hemodynamic monitoring, blood glucose monitoring for hypoglycemia, "
    "and fluid restriction to prevent acute pulmonary edema are mandatory in all intensive care units."
)

# 1. Parent splitter (1200 chars = full clinical context)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=100)
parents = parent_splitter.split_text(clinical_manual_text)

# 2. Child splitter (200 chars = sharp, accurate search key)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

demo_database = []
for p_idx, parent in enumerate(parents):
    children = child_splitter.split_text(parent)
    for c_idx, child in enumerate(children):
        chunk_id = f"chunk_{p_idx}_{c_idx}"
        demo_database.append({
            "chunk_id": chunk_id,
            "search_key_child": child,      # What gets embedded and searched
            "llm_context_parent": parent    # What gets handed to GPT-4o!
        })

print(f"Created {len(demo_database)} searchable Child Chunks from 1 Parent Block:\n")
for item in demo_database:
    print(f"[{item['chunk_id']}] SEARCH KEY (Child): \"{item['search_key_child']}\"")
    print(f"   └──► Preserved Parent Context: {len(item['llm_context_parent'])} chars (Contains pregnancy warnings!)\n")


---
## 🧭 Module 3: Vector Embeddings in Plain English (00:18 – 00:28)
### *GPS Coordinates for Human Thought*

### The Intuitive Concept:
How do computers know that *"puppy"* and *"dog"* mean the same thing, but *"bank"* (river) and *"bank"* (money) mean different things?

- An embedding model is a function that takes any piece of text and assigns it **GPS coordinates** in high-dimensional space (e.g., 3,072 numbers).
- In this space:
  - *"high blood pressure"* and *"hypertension"* are plotted **right next to each other** on the map.
  - *"chocolate cake recipe"* is plotted on a completely different continent!

```
           Medical Continent                           Cooking Continent
   ┌────────────────────────────────┐            ┌────────────────────────────┐
   │ • "elevated blood sugar"       │            │ • "chocolate cake recipe"  │
   │ • "gestational diabetes"       │            │ • "brownie baking temp"    │
   │ • "glycemic targets"           │            │ • "confectionery sugar"    │
   └────────────────────────────────┘            └────────────────────────────┘
```

### Measuring Distance: Cosine Similarity
- If two arrows point in the exact same direction $\to$ Similarity = **1.0** (identical meaning).
- If two arrows point at 90 degrees $\to$ Similarity = **0.0** (totally unrelated).
- If vectors are normalized to length 1, cosine similarity is just the **dot product** (multiplying corresponding numbers and summing them up).


In [ ]:
# LIVE DEMO: Proving Semantic Similarity with ZERO Word Overlap
from openai import OpenAI
import numpy as np

client = OpenAI()

def get_vec(text):
    resp = client.embeddings.create(input=[text], model="text-embedding-3-large")
    vec = np.array(resp.data[0].embedding, dtype=np.float32)
    return vec / np.linalg.norm(vec)

query = "elevated blood sugar levels during pregnancy"
sentence_a = "Gestational diabetes screening guidelines and glycemic targets"
sentence_b = "Drinking sweet soda with sugar can elevate blood sugar during sports matches"

v_query = get_vec(query)
v_a = get_vec(sentence_a)
v_b = get_vec(sentence_b)

sim_a = float(np.dot(v_query, v_a))
sim_b = float(np.dot(v_query, v_b))

print(f"User Query: \"{query}\"\n")
print(f"Candidate A: \"{sentence_a}\"")
print(f"  • Common words with query: ZERO words!")
print(f"  • Cosine Similarity: {sim_a:.4f}  <── HIGH! The model understands medical meaning.\n")

print(f"Candidate B: \"{sentence_b}\"")
print(f"  • Common words with query: 'elevated', 'blood', 'sugar', 'during' (4 matches!)")
print(f"  • Cosine Similarity: {sim_b:.4f}  <── LOWER! The model knows sports != obstetrics.\n")


---
## 🎯 Module 4: Why Vector Search Fails & Why You Need Hybrid (00:28 – 00:38)
### *The Fuzzy Artist (Vector) + The Exact Accountant (BM25)*

### The Fatal Blindspot of Vector Search:
Vector search is **fuzzy and conceptual**. 
If a user searches for an exact alphanumeric code or a specific drug:
- Query: *"What is the adult dose of Pyrazinamide?"*
- Vector Search says: *"Pyrazinamide is an antibiotic for Tuberculosis... Isoniazid and Rifampicin are also TB antibiotics! They all live in the same neighborhood. Here is a chunk about Isoniazid!"*
- **In software and medicine, this is unacceptable**. You cannot substitute parts, error codes (`ERR-502`), or medications based on "conceptual vibe".

---

### The Solution: Hybrid Search (Dense + BM25)
1. **Dense Search (OpenAI / Qdrant)**: Finds conceptual matches and handles synonyms.
2. **BM25 (Keyword Search)**: An ultra-fast exact-match index that rewards **rare words** (like *Pyrazinamide*, *RTX-4090*, *CVE-2024-1234*).

---

### How to Merge Them: Reciprocal Rank Fusion (RRF)
How do you merge a vector score (e.g. `0.84`) with a BM25 score (e.g. `24.6`)?
You **cannot** add them together. BM25 scores explode on long queries.

**RRF (Cormack et al., 2009)** uses a brilliantly simple rule: **Ignore the raw scores, look only at the RANK position!**
$$	ext{RRF Score} = rac{1}{60 + 	ext{Rank}_{	ext{Dense}}} + rac{1}{60 + 	ext{Rank}_{	ext{BM25}}}$$

- If a chunk is #1 in Dense, but missing in BM25: Score = $rac{1}{61} + 0 = \mathbf{0.0163}$
- If a chunk is #3 in Dense AND #3 in BM25: Score = $rac{1}{63} + rac{1}{63} = \mathbf{0.0317}$
- **The consensus document wins every time!**


In [ ]:
# LIVE DEMO: BM25 Exact Matching + Reciprocal Rank Fusion
from rank_bm25 import BM25Okapi

sample_docs = [
    "Isoniazid and Rifampicin intensive therapy for pulmonary tuberculosis in adults.",
    "Pyrazinamide adult dosing: 25 mg/kg daily as part of the four-drug intensive HRZE regimen.",
    "Ethambutol ocular toxicity screening and visual acuity monitoring.",
    "Hypertension management: Amlodipine 5mg titrated to 10mg daily."
]

tokenized = [doc.lower().replace(".", "").replace(":", "").split() for doc in sample_docs]
bm25 = BM25Okapi(tokenized)

query = "adult dose of Pyrazinamide"
scores = bm25.get_scores(query.lower().split())

print("--- BM25 KEYWORD SCORES ---")
for doc, score in zip(sample_docs, scores):
    print(f"  Score: {score:5.2f} | \"{doc[:60]}...\"")

# Live RRF Fusion demonstration
def rrf_merge(dense_ranks, bm25_ranks, K=60):
    combined = {}
    for r, doc in enumerate(dense_ranks):
        combined[doc] = combined.get(doc, 0.0) + 1.0 / (K + r + 1)
    for r, doc in enumerate(bm25_ranks):
        combined[doc] = combined.get(doc, 0.0) + 1.0 / (K + r + 1)
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)

# Suppose Dense preferred Doc A, but BM25 preferred Doc B:
dense_list = ["Doc_A (General TB)", "Doc_B (Pyrazinamide)", "Doc_C (Ethambutol)"]
bm25_list  = ["Doc_B (Pyrazinamide)", "Doc_D (Unrelated)", "Doc_A (General TB)"]

print("\n--- RRF FUSED RANKINGS ---")
fused = rrf_merge(dense_list, bm25_list)
for rank, (doc, score) in enumerate(fused, 1):
    print(f"  Rank #{rank}: {doc:22} | Merged RRF Score: {score:.5f}")


---
## 🔬 Module 5: The Two-Stage Filter (Bi-Encoder vs Cross-Encoder) (00:38 – 00:46)
### *Why Bi-Encoders are Blind to the Word "NOT"*

### The Problem: Negation Blindness
In Dense Vector Search (Bi-Encoder), query and document are turned into vectors **separately**:
$$\mathbf{q} = 	ext{Embed}(	ext{Query}), \quad \mathbf{d} = 	ext{Embed}(	ext{Doc})$$

Because vectors are averages of all words, the tiny word **"NOT"** carries almost zero weight.
- Query: *"High blood pressure treatment with NO history of diabetes"*
- Bi-Encoder: High similarity with *"Diabetic hypertension treatment guidelines"*!

---

### The Solution: Cross-Encoder Reranking
In a Cross-Encoder, the Query and Document are glued together and read by the transformer **at the exact same time**:
$$	ext{Input} = 	ext{[CLS] Query Tokens [SEP] Document Tokens [SEP]}$$

Every word in the query directly looks at every word in the document through **cross-attention**.
- When the cross-encoder sees *"NO history of diabetes"* next to *"For diabetic patients"*, it immediately detects the contradiction and penalizes the score!

```
TWO-STAGE PIPELINE:
[15,000 Documents] 
      │
      ▼ (Stage 1: Bi-Encoder + BM25 — Fast, 20ms, takes 15,000 down to 15)
[Top 15 Candidates]
      │
      ▼ (Stage 2: Cross-Encoder — Deep attention, 120ms on CPU, takes 15 down to 3)
[Top 3 Pristine Chunks] ──► Sent to LLM!
```


In [ ]:
# LIVE DEMO: Running local Cross-Encoder (ms-marco-MiniLM on CPU)
from sentence_transformers import CrossEncoder

# Runs locally on your laptop CPU in ~100ms!
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "Treatment of hypertension in pregnant patients"

candidates = [
    # Candidate 1: High keyword overlap, but medically CONTRAINDICATED in pregnancy!
    "ACE-inhibitors like enalapril are highly effective for hypertension but cause fetal renal failure and are contraindicated in pregnancy.",
    
    # Candidate 2: Truly indicated for pregnancy
    "Labetalol and Methyldopa are recommended first-line antihypertensive agents during pregnancy.",
    
    # Candidate 3: Unrelated adult males
    "General guidelines for managing stage 1 hypertension in adult males."
]

pairs = [(query, c) for c in candidates]
scores = reranker.predict(pairs)

print(f"Query: \"{query}\"\n")
print("RERANKER RELEVANCE LOGIT SCORES (Sorted by true relevance):")
for doc, score in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True):
    print(f"  Score: {score:+6.3f} | \"{doc[:85]}...\"")


---
## 🛡️ Module 6: Safety Guardrails & The "Bouncer" (00:46 – 00:54)
### *Stopping Hallucinations BEFORE Calling the LLM*

### The "Forced Answer" Problem:
Standard RAG assumes the search engine always finds relevant info.
If an employee asks your internal company HR bot:
> *"What is the best recipe for chocolate cake?"*

Vector search will find the 5 closest documents in your HR wiki (perhaps the cafeteria menu or birthday party expense policy). It hands them to GPT-4o, and GPT-4o tries to be helpful and hallucinates a recipe!

---

### The Fix: The Cross-Encoder Confidence Gate (The Bouncer)
Notice the scores output by `ms-marco-MiniLM`:
- **Great in-domain match**: $+1.0$ to $+6.0$
- **Weak / vague match**: $-2.0$ to $-3.5$
- **Total garbage / off-topic** (FIFA, Cake, Blockchain): **$-7.0$ to $-11.0$**!

We set a **Confidence Gate at `-4.0`**:
$$	ext{If } 	ext{Top-1 Score} < -4.0 \implies 	ext{STOP! Refuse immediately. Do NOT call GPT-4o!}$$

```
Logit Score Spectrum:
◄──────────────────────────────────────────────────────────────────────────────────►
-11.0         -9.0         -7.5       -5.4     -4.0      -2.0        0.0       +4.5
  │            │            │          │        │         │           │         │
Blockchain    Cake         FIFA       Table  THRESHOLD Diarrhoea   Clean TB  Clean HTN
[Off-topic]  [Off-topic]  [Off-topic] [Frag]    │      [Medical]   [Medical] [Medical]
                                                │
                               ◄─── REJECT ─────┼───── PASS ───►
```
- **Saves Money**: Zero LLM API tokens spent on off-topic questions.
- **Zero Hallucination**: Impossible to hallucinate because the generator is never invoked.


In [ ]:
# LIVE DEMO: The Confidence Gate in Action
THRESHOLD = -4.0

queries_to_test = [
    ("What are standard treatment guidelines for tuberculosis?",
     "Standard treatment for new pulmonary tuberculosis consists of 2 months of HRZE daily.",
     "VALID IN-DOMAIN QUERY"),
    
    ("How do I bake a chocolate cake at home?",
     "Antimicrobial guidelines: surgical prophylaxis and sterile operating theatre protocol.",
     "OFF-TOPIC / GARBAGE QUERY")
]

for q, doc, label in queries_to_test:
    score = float(reranker.predict([(q, doc)])[0])
    passed = score >= THRESHOLD
    action = "✅ PASS ──► Send to GPT-4o" if passed else "🚫 REFUSE (Score < -4.0) ──► Stop! Save $ & prevent hallucination"
    
    print(f"[{label}]")
    print(f"  User Query: \"{q}\"")
    print(f"  Top Match Score: {score:+.3f} (Threshold: {THRESHOLD})")
    print(f"  Decision: {action}\n")


---
## 📊 Module 7: How to Measure Success & Team Q&A (00:54 – 01:00)
### *The TruLens RAG Triad: Stop "Vibe-Checking" Your Pipeline*

When teams build RAG, they often test 3 questions by hand, say *"looks good to me"*, and deploy. Two weeks later, users report hallucinations.

### The 3 Golden Metrics (The RAG Triad):
```
                              User Query (Q)
                             ┌──────────────┐
                             │      Q       │
                             └──────┬───────┘
                     Context        │        Answer
                    Relevance       │      Relevance
                   (retriever)      │     (generation)
                        │           │          │
                        ▼           │          ▼
                 ┌──────────────┐   │   ┌─────────────┐
                 │ Context (C)  │◄──┴───┤  Answer (A) │
                 └──────────────┘       └─────────────┘
                        ▲                      │
                        └──────────────────────┘
                              Groundedness
                             (faithfulness)
```

1. **Context Relevance**: Did the retriever find chunks that actually answer $Q$? (Measures your search quality).
2. **Groundedness / Faithfulness**: Is every claim in $A$ 100% supported by $C$? (Measures hallucination rate).
3. **Answer Relevance**: Did $A$ actually answer the user's specific question?

---

## 🎯 5 Core Takeaways for the Team:
1. **Never use single-size chunks**: Use **Parent-Child Chunking** (search short child chunks, generate from rich parent chunks).
2. **Always combine Dense + BM25**: Vectors find synonyms; BM25 finds exact part numbers, drug names, and error codes.
3. **Always use a Cross-Encoder Reranker**: It takes 100ms on CPU and catches negations ("NOT", "except") that bi-encoders miss.
4. **Always put a Confidence Gate at the door**: Refuse queries below threshold (e.g. `-4.0`) before calling the LLM to save money and stop hallucinations.
5. **Enforce JSON citations**: Verify programmatically that every cited source exists in the retrieved context.

---
### ❓ Open Floor for Questions & Discussion!
